# Building a new combinatorial algebra with 2x2 matrices

This notebook is meant as a guided example of how a user can create a new algebra from a mathematical idea.

In [2]:
import sys
from pathlib import Path

repo_root = None
for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
    if (candidate / "algcom").exists():
        repo_root = candidate
        break
if repo_root is not None:
    sys.path.insert(0, str(repo_root))

import numpy as np
from algcom import Algebra, SparseVector
from fractions import Fraction

## 1. The "sparse" space of square matrices.

We want to work with 2x2 matrices with the usual multiplication. Let us remember that a matrix is a linear combination of the four elements $e_{11}$, $e_{12}$, $e_{21}$ and $e_{22}$. For instance, we can write the identity matrix as the sum of two vectors. 

In [3]:
e11 = (1,1)
e12 = (1,2) 
e21 = (2,1)
e22 = (2,2)

identity = SparseVector(e11) + SparseVector(e22)
print(f"{identity}")

⟅(1, 1) + (2, 2)⟆


The complex numer $\mathtt{i}$, is given by $\mathtt{i} = e_{21} - e_{12}$.

In [4]:
im = SparseVector(e21) - SparseVector(e12)
print(f"Imaginary unit: {im}")

Imaginary unit: ⟅-(1, 2) + (2, 1)⟆


Let us see how linear combinations are evaluated. Notice that we are using rationals for the coefficients.

In [5]:
print(f"1 + 2i = {identity + 2*im}")
print(f"(1/3) + (1/2)i = {identity*Fraction(1,3) + im*Fraction(1,2)}") 

1 + 2i = ⟅(1, 1) - 2 (1, 2) + 2 (2, 1) + (2, 2)⟆
(1/3) + (1/2)i = ⟅2 (1, 1) - 3 (1, 2) + 3 (2, 1) + 2 (2, 2)⟆/6


The printing of a linear expression is made such that the coefficients inside the delimiters ⟅...⟆ were always integers. The common denominator is expelled outside, such as the $...⟆/6$ in the last expression.

## Multiplication and defining an algebra 

In the space of matrices, and this means all the matrices, the produc is given by the rule
$$ e_{i,j} \cdot e_{k,l} = \delta_{j,k} e_{i,l}.$$
Here $\delta$ is Kronecker's delta and $i,j,k,l$ are any positive integers.

Let us implement the rule and execute some examples. 

Remember, the rule is given at the level of basis' elements!

In [6]:
mult = lambda left,right : (left[0],right[1]) if left[1]==right[0] else 0 

print(f"Is {e12} = {mult(e11,e12)}?")
print(f"And {mult(e21,e22)} must be zero.")

Is (1, 2) = (1, 2)?
And 0 must be zero.


We need now to lift this (set)-product into a bilinear multiplication. The class Algebra handles that bilinearity without much of a hassle for the user. 

A comment, though. Since the package was thought for combinatorial objects, the product should return a either list of elements of the basis or a sparse vector.
In this case, the product generates only one element of the basis.

In [7]:
mult = lambda left,right : [ (left[0],right[1]) ] if left[1]==right[0] else []

sqmat = Algebra.from_rule( mult , one = identity ) 

And that's it. The moment of the truth has arrived.

In [8]:
print(f"1 + i*i = {identity + sqmat.m(im,im)}")

1 + i*i = 0


And we can do more than that. We can compute for instance exponentials and logarithms without further implementation.
In the next example, we compute up to three digits.
$$ \exp\left\{ \log{2} + \frac{\pi}{4}\mathtt{i} \right\}  = \sqrt{2} + \sqrt{2}\mathtt{i} $$

In [37]:
log2 = Fraction(np.log(2)).limit_denominator()
angle = Fraction(np.pi/4).limit_denominator()

print(f"log(2*1)={sqmat.logarithm(2*identity,order=750)}" )

operando = sqmat.unit(log2) + angle * im 
result =  sqmat.exponential(  operando , order=20)
print(f"exp( log(2) + pi/4 * i )={result}")

print(f"Geomentric series convergence: {sqmat.geometric(sqmat.unit(Fraction(1,2)),order=9)}")

log(2*1)=⟅692 (1, 1) + 692 (2, 2)⟆/1000
exp( log(2) + pi/4 * i )=⟅1414 (1, 1) - 1414 (1, 2) + 1414 (2, 1) + 1414 (2, 2)⟆/1000
Geomentric series convergence: ⟅1023 (1, 1) + 1023 (2, 2)⟆/512


## Other linear constructions



## Optional: Class structure for the elements of the basis

The package was designed for its full-use potential using inheritance and polymorphism. 

At this point, the object "algebra" knows how to multiply elements, and it also knows what the unit element is.

## 3. Multiply them

The algebra object uses the product rule we gave it.

In [9]:
ab = algebra.m(a, b)
ba = algebra.m(b, a)

print(ab)
print(ba)

NameError: name 'algebra' is not defined

## 4. Use the generic algebra operations

Once the algebra is defined, you can ask for powers, exponentials, and logarithms.

In [ ]:
square_a = algebra.power(a, 2)
exp_a = algebra.exponential(a, order=3)

print(square_a)
print(exp_a)

## 5. A comment on the design

The important idea is that you do not need to write a new class for every algebra. You define the product rule, specify the unit, and the rest of the algebraic operations become available.

## 6. A small recipe for a new algebra

1. Pick a mathematical object and a product.
2. Choose a unit element.
3. Write the product rule as a function.
4. Build an Algebra instance with Algebra.from_rule.
5. Use the resulting object for multiplication, powers, exponentials, and logarithms.